## Cell 0. API keys

Paste your keys between the quotes below and run this cell before anything
else. Leave a line as `""` to use whatever is already exported in the
environment instead.

**Two things worth knowing before you paste.** This notebook is regenerated by
`build_q1_nb.py`, which rewrites every cell from source -- so a key typed here
is lost on the next rebuild. And a key typed here is saved inside the `.ipynb`
file, where it can reach git or a shared copy. For a key you intend to keep,
put it in `fourarm/env/keys.local.env` instead, which this cell reads
automatically and which is gitignored and never regenerated.

In [ ]:
# --- Cell 0. API keys. Run first. -------------------------------------------
import os, pathlib

# PASTE BETWEEN THE QUOTES. Leave "" to fall back to the environment or to
# env/keys.local.env.
KEYS = {
    "OPENAI_API_KEY": "",
    "GEMINI_API_KEY": "",
    "ANTHROPIC_API_KEY": "",
}

# An EMPTY value must never be written into the environment. Assigning ""
# unconditionally would blank a key that is already exported correctly, and
# the failure -- a 401 from a variable that is set but empty -- reads nothing
# like "you left the placeholder alone".
for _name, _value in KEYS.items():
    if _value.strip():
        os.environ[_name] = _value.strip()

# The persistent alternative. Same KEY=value format as env/models.env, one
# per line, # for comments. Read only for names not already set, so anything
# pasted above and anything already exported both win over the file.
_here = pathlib.Path.cwd()
_root = next((c for c in [_here] + list(_here.parents)
              if (c / "out").is_dir() and (c / "experiments").is_dir()), None)
_local = _root / "env" / "keys.local.env" if _root else None
if _local and _local.exists():
    for _line in _local.read_text().splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _, _v = _line.partition("=")
        _k, _v = _k.strip(), _v.strip().strip("\'\"")
        if _v and not os.environ.get(_k):
            os.environ[_k] = _v

# Report presence, NEVER the value. Printing a key would write it into the
# notebook's saved output, which is the same leak as pasting it into a cell
# and is easier to do by accident.
#
# The report loops over KEYS, so EVERY key the notebook can use needs a row
# there even when it is only ever supplied by keys.local.env. A name missing
# from KEYS still loads from the file, but silently, and a key that loads
# without being reported is indistinguishable from one that did not load.
for _name in KEYS:
    _set = bool(os.environ.get(_name))
    print("%-18s %s" % (_name, "set" if _set else "NOT SET"))
if _local:
    print("%-18s %s" % ("keys.local.env",
                        "read" if _local.exists() else "absent (optional)"))

# Experiment 2, Q1: Derivation

**When the text omits the capability-relevant quantity, can the model obtain it
from the scene?**

Read at rung **N0** only. The other rungs belong to Q3.

Every cell is independently runnable and idempotent. No cell overwrites a paid
run: the runners resume into their output file and skip trials already
answered. Cells that spend money print the call count and refuse to proceed
until `CONFIRM_SPEND` is set to that exact number.

Run this notebook with the working directory set to `fourarm/`, or anywhere
below it -- cell 1 finds the root itself.

In [ ]:
# --- Cell 1. Setup. No model calls. -----------------------------------------
import collections, csv, datetime, hashlib, json, math, os, pathlib, sys

# Find the package root: the directory holding out/ and experiments/.
here = pathlib.Path.cwd()
ROOT = None
for cand in [here] + list(here.parents):
    if (cand / "out").is_dir() and (cand / "experiments").is_dir():
        ROOT = cand
        break
if ROOT is None:
    raise SystemExit("run this from fourarm/ or below: no out/ + experiments/ found")
for p in (str(ROOT), str(ROOT / "ycb")):
    if p not in sys.path:
        sys.path.insert(0, p)

# RE-IMPORT, never reuse. Python caches modules in sys.modules, so running
# this cell a second time in a live kernel keeps whatever was on disk the
# FIRST time it ran. While the ex2 modules are being edited alongside the
# notebook that is a trap: the kernel holds the old vocabulary, and the
# failure surfaces cells later as a design-check assertion naming a face
# that no longer exists, which reads like a code error and is not one.
#
# Dropping the entries and importing fresh is used rather than
# importlib.reload because these modules import each other, and reload
# leaves a half-updated graph unless the order is exactly right.
for _stale in [m for m in list(sys.modules)
               if m.startswith(("experiments.ex2", "analysis.ex2"))
               or m in ("ycb_objects",)]:
    del sys.modules[_stale]

# WHICH KERNEL THIS IS, checked before the first project import.
#
# The very next line reaches core.decision.state_builder through
# mancheck -> vlm_allocator, and that imports numpy; visibility.py, in cell
# 3, needs PIL and scipy. On a kernel without them the notebook dies forty
# lines deep inside somebody else's module with "No module named 'numpy'",
# which reads as a broken repository rather than as a kernel picked from a
# list of six. Checked here, where the answer is one sentence.
_missing = []
for _m in ("numpy", "PIL", "scipy"):
    try:
        __import__(_m)
    except ImportError:
        _missing.append(_m)
if _missing:
    _venv = ROOT.parent / ".venv" / "bin" / "python"
    raise SystemExit(
        "WRONG KERNEL.\n"
        "  This kernel is  %s\n"
        "  and it has no %s.\n"
        "  Use instead     %s\n"
        "  In VS Code: Select Kernel, then Python Environments, then the\n"
        "  interpreter at that path. It is the only one in this tree with\n"
        "  ipykernel AND numpy, PIL and scipy. Several unrelated kernels are\n"
        "  registered on this machine and any of them will get this far and\n"
        "  then fail."
        % (sys.executable, ", ".join(_missing), _venv))

from core.cell import cell_config as C
from core.decision import model_registry as MR
from experiments.ex2 import grade as G
from experiments.ex2 import labels as L
from experiments.ex2 import mancheck as MC
from experiments.ex2 import prompts as P
from experiments.ex2 import run as R
from experiments.ex2 import solo as S
from experiments.ex2 import transforms as T
from experiments.ex2 import visibility as VIS
from analysis.ex2.ex2_stats import newcombe, paired_mean_ci, spans_zero, wilson
# The notebook machinery: loaders, the share definition, the paired
# contrast and the spend gate. In a module rather than in this cell so
# that Q2 and Q3 use the same ones rather than a second copy, and so
# that harness/h_ex2_q_common.py can pin them. What stays in the cells
# is what is a DECISION: the models, the rung, the conditions, the
# usable rule, each cost, and every CONFIRM_SPEND.
from analysis.ex2.ex2_q_common import (Outputs, answered,       # noqa
                                       coupling, fmt, full_flip_count,
                                       is_franka, keep_analysable,
                                       load_run, paired_delta,
                                       paired_diffs, pct,
                                       provenance_row, run_meta,
                                       sha256, share_at, share_counts,
                                       show, spend_gate)

# --- paths ------------------------------------------------------------------
CAPTURES = ROOT / "out" / "ex2_capture_block"
RUNS     = ROOT / "runs"
TABLES   = ROOT / "tables" / "ex2_q1"
FIGURES  = ROOT / "figures" / "ex2_q1"
for d in (RUNS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# --- constants, every one read from a source of truth ------------------------
RUNG        = "N0"                       # Q1 is read here and nowhere else
PREFERENCE  = "franka"
CONDITIONS  = ("congruent", "congruent_face")
REPEATS     = 1
FACES       = P.RESTING_FACES            # small_face, large_face
LABEL       = "ycb_block"

FRANKA_MAX  = C.ARM_TYPES["franka"]["max_grasp_m"]
UR_MAX      = C.ARM_TYPES["ur10"]["max_grasp_m"]
DIMS        = T.DIMS_M[LABEL]
FACTS       = T.POSE_FACTS_BY_LABEL[LABEL]

# The registry has no hardcoded model list: aliases() reads FOURARM_MODELS.
try:
    ALIASES = MR.aliases()
except Exception as exc:
    ALIASES = []
    print("model registry unavailable (%s); set MODELS by hand below" % exc)
# THREE models since 2026-08-27. claude-sonnet-5 was added because the
# design needs a third model that CLEARS the two-way face probe: with two
# models, a single failure at cell 5b leaves one, and one model cannot show
# that a result is a property of models rather than of this one model.
# It is not here for being the most capable available; see env/models.env.
#
# claude_md RATHER THAN claude. Same model, claude-sonnet-5, at effort
# medium instead of the API default of high. At the default it read the
# two-way face probe at 65 percent against gpt's 95 and gemini's 100, and
# it failed by BIAS rather than blindness: large_face on 75 percent of
# trials, 90 percent right when the block lies flat and 40 percent when it
# stands. Deliberation is how a prior like "blocks lie flat" gains weight,
# so lower effort is the move that fits the failure. Cell 5b is what tests
# it. The default-effort runs stay on disk under the alias "claude".
#
# gpt_hi RATHER THAN gpt. Same model, gpt-5.6-terra, at reasoning_effort
# high instead of low. Claude runs at the Anthropic default effort of high
# and Gemini Flash exposes no effort control at all, so gpt at low made the
# one model with the LEAST test-time compute the yardstick for the other
# two. Effort is still not matched across providers and cannot be -- that
# stays in Limitations -- but the reasoning models are now on the same
# nominal tier.
#
# The low-effort runs are NOT deleted. runs/ex2_q1_cue2way_gpt_r*.jsonl
# record gpt at reasoning_effort low over this same sample and stay on disk
# as the evidence for what effort was worth here: 95 percent at low. A
# separate alias rather than an edit to GPT_PARAMS is what makes those rows
# still readable, which is the reason env/models.env gives for gpt_hi
# existing at all.
#
# Named rather than taken wholesale from ALIASES. FOURARM_MODELS also lists
# qwen and gpt_hi, and a run's model set must be a decision recorded here,
# not whatever the registry happens to carry. The fallback keeps the same
# three so a registry failure cannot silently shrink the design.
_WANT = ("gpt_hi", "gemini", "claude_md")
MODELS = tuple(a for a in ALIASES if a in _WANT) or _WANT
if set(MODELS) != set(_WANT):
    print("WARNING: %s requested, %s available from the registry. Every "
          "table below is per model, so a missing one narrows the design "
          "rather than breaking it -- but say so in the chapter."
          % (list(_WANT), list(MODELS)))

# --- credentials: reported, not assumed -------------------------------------
# Until 2026-08-27 a hand-added launcher cell started JupyterLab in a browser
# and refused to launch when a key was missing. That cell is gone: it spawned
# a NEW server every time it ran, which is how eight of them accumulated, each
# serving its own in-memory copy of this notebook, so an edit on disk could be
# invisible in the tab you were typing in. VS Code runs the kernel directly
# and needs no launcher -- but the key check it performed was worth keeping,
# so it lives here.
#
# This REPORTS rather than raises. Cells 1-5 and every analysis cell make no
# model calls and must stay runnable with no key at all. What it buys is
# learning about a missing key now instead of at cell 6, part-way into a run.
#
# The usual cause is launching VS Code from Finder or the Dock, which does not
# inherit a login shell, so a key exported in .zshrc is absent here while
# present in any terminal. The message below says so, because the symptom
# otherwise looks like a broken registry.
#
# key_var is read from the registry, never hardcoded: models.env lets each
# alias name its own variable, and a hardcoded OPENAI_API_KEY would check the
# wrong one the moment that is used.
MISSING_KEYS = []
for _alias in MODELS:
    try:
        _var = MR.describe(_alias)["key_var"]
    except Exception as _exc:
        MISSING_KEYS.append("%s: %s" % (_alias, _exc))
        continue
    if not os.environ.get(_var):
        MISSING_KEYS.append("%s: %s is not set" % (_alias, _var))

# Output paths travel together in one object, so a notebook cannot end up
# with a root and a tables directory that disagree. Rebound to bare names
# because every call site below reads better as write_csv(...) than as
# OUT.write_csv(...), and because leaving those call sites untouched is
# what made this extraction verifiable against the tables already on disk.
OUT = Outputs(ROOT, TABLES, FIGURES)
rel, write_csv = OUT.rel, OUT.write_csv

print("root        ", ROOT)
print("captures    ", CAPTURES.relative_to(ROOT), "(exists:", CAPTURES.is_dir(), ")")
print("rung        ", RUNG, " preference", PREFERENCE, " repeats", REPEATS)
print("models      ", MODELS, " (registry knows: %s)" % (ALIASES or "nothing"))
print("prompt ver  ", P.EX2_PROMPT_VERSION)
# Printed, not assumed. If a stale kernel ever slips past the re-import
# above, this is the line that shows it, at the top of the run rather than
# in an assertion twenty cells later.
print("faces       ", FACES, " chance %.1f%%" % (100.0 / len(FACES)))
if MISSING_KEYS:
    print("api keys     MISSING -- analysis runs, model calls will not:")
    for _m in MISSING_KEYS:
        print("               ", _m)
    print("             launch VS Code from a shell that exports them:")
    print("               open -a 'Visual Studio Code' <repo>")
else:
    print("api keys     present for %s" % (", ".join(MODELS),))
print()
print("block           %.3f x %.3f x %.3f m"
      % (DIMS["height"], DIMS["width"], DIMS["depth"]))
print("franka opens to  %.3f m   ur opens to %.3f m" % (FRANKA_MAX, UR_MAX))
print("resting faces   %s" % (FACES,))
for f in FACES:
    print("   %-11s needs %.3f m" % (f, FACTS[f]["grasp_m"]))

## Frame probe: does the `named` glossary leak where a face IS stated?

**Makes model calls.** Two conditions, three models, one repeat, about 400 calls,
roughly 12 minutes with one process per model.

`congruent` and `congruent_face` both state `resting_face` in the state. The
`named` glossary additionally says the extents were "measured standing on its
smallest face", which describes `small_face`. If the models are reading that
phrase as a pose claim even when a pose is given, the supplied-face conditions
are partly text-following too, and the reference lines they anchor need
qualifying. If nothing moves, the confound is confined to `dims` and the
correction is one subsection.

Existing `named` runs to compare against are already on disk at three repeats:
`ex2_q1_congruent_N0.jsonl` and `ex2_q1_congruent_face_N0.jsonl`.

In [ ]:
# --- Frame probe. PAID. -----------------------------------------------------
from experiments.ex2 import launch as L

PROBE_TAG   = "q1_frameprobe_N0_extents"
PROBE_CONDS = ("congruent", "congruent_face")

jobs = L.per_model(
    ("gpt", "gemini", "claude"),
    capture_dir = str(CAPTURES),
    out_dir     = RUNS,
    tag         = PROBE_TAG,
    conditions  = PROBE_CONDS,
    rungs       = (RUNG,),                 # N0, from cell 1
    dims_frames = ("extents",),
    repeats     = 1,
)

print("about to buy:")
for j in jobs:
    print("   %-8s -> %s" % (j["name"], pathlib.Path(j["kwargs"]["out_path"]).name))
print("   %s x %s x 1 repeat" % (PROBE_CONDS, ("gpt", "gemini", "claude")))
print()

handles = L.start(jobs)
print("started, nothing blocks. run the next cell when they finish.")

## Frame probe read-out

No model calls. `extents` against the `named` runs already on disk.

**The anchor is what to read.** Both conditions state the true face, so opening
accuracy should already be high under both frames and would not separate the two
explanations. What separates them is whether the numbers MOVE when the only thing
that changed is a phrase the condition has made redundant.

In [ ]:
# --- Frame probe read-out. No model calls. ----------------------------------
rows = L.combine(handles)

NAMED = {"congruent":      RUNS / "ex2_q1_congruent_N0.jsonl",
         "congruent_face": RUNS / "ex2_q1_congruent_face_N0.jsonl"}

def _load(path):
    if not pathlib.Path(path).exists():
        return []
    return [json.loads(l) for l in open(path)]

def _stats(recs, cond, model):
    r = [x for x in recs if x.get("condition") == cond
         and x.get("model") == model and not x.get("error")]
    op = [x for x in r if x.get("opening_needed_m") is not None]
    ok = sum(1 for x in op
             if abs(x["opening_needed_m"] - x["true_grasp_m"]) < 0.006)
    sml = sum(1 for x in op if abs(x["opening_needed_m"] - 0.050) < 0.006)
    ar = [x for x in r if x.get("arm")]
    fk = sum(1 for x in ar if "franka" in x["arm"])
    pct = lambda k, n: (100.0 * k / n) if n else None
    return len(r), pct(ok, len(op)), pct(sml, len(op)), pct(fk, len(ar))

ext = list(rows)
hdr = ("condition", "model", "frame", "n", "opening ok%",
       "reports .050%", "franka%")
out = []
for cond in PROBE_CONDS:
    for m in MODELS:
        for frame, recs in (("named", _load(NAMED[cond])), ("extents", ext)):
            n, ok, sml, fk = _stats(recs, cond, m)
            f = lambda v: "--" if v is None else "%.1f" % v
            out.append([cond, m, frame, n, f(ok), f(sml), f(fk)])

w = [max(len(str(r[i])) for r in [list(hdr)] + out) for i in range(len(hdr))]
print("  ".join(str(h).ljust(w[i]) for i, h in enumerate(hdr)))
print("  ".join("-" * w[i] for i in range(len(hdr))))
for r in out:
    print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(r)))

print()
print("=" * 70)
print("WHAT THE TWO OUTCOMES MEAN, stated before the numbers are read")
print("=" * 70)
print("  NOTHING MOVES     the glossary phrase is inert once a face is")
print("                    stated. The confound is confined to dims. Q1's")
print("                    reference lines and all of Q2 stand, and the")
print("                    correction is one subsection about dims.")
print()
print("  SOMETHING MOVES   the phrase is read as a pose claim even when a")
print("                    pose is given. The supplied-face conditions are")
print("                    partly text-following, and every reference line")
print("                    drawn from them needs qualifying.")